# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Damasoumana1/flyrank-ml-internship-july2026/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My ML Task Type: Ranking

Task type: Ranking
My lane is a ranking problem because the goal is to prioritize webpages for content review rather than simply classify them as good or bad. The system should assign each webpage a priority based on observable search and engagement signals, then rank webpages from the highest-priority opportunities to the lowest-priority ones.

This ranking supports a content team's decision about which webpages should be reviewed first for refresh, optimization, monitoring, or further investigation.

In [4]:
import os
import subprocess

REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    ], check=True)

os.chdir(REPO_DIR)

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("ML task type: Ranking")

Rows: 30000
Columns: 44
ML task type: Ranking


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target or Proxy

The target is an observed **declining trend** for a webpage.

A page is labeled as declining when its observed `trend_direction` is `down`. This is an observed outcome in the starter dataset, rather than a product decision or a manually assigned refresh flag.

The model would use observable pre-decision signals such as impressions, average position, CTR, content age, and update recency to learn patterns associated with declining pages.

This target is used as a proxy for identifying webpages that may deserve review. It does not mean that a page labeled as declining will necessarily benefit from a refresh.

In [5]:
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Target: observed declining trend")
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Target: observed declining trend
Declining rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success Metric

The primary success metric is **Precision@50**.

Precision@50 measures the proportion of the 50 highest-ranked webpages that are actually labeled as declining.

A higher Precision@50 means that more of the pages prioritized for review correspond to the observed declining-trend target.

For this task, a higher Precision@50 is better because the content team's review capacity is limited, so the ranking should place as many relevant pages as possible near the top of the priority list.

In [6]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Success metric: Precision@50")
print("A higher value means more relevant pages in the top 50.")

Success metric: Precision@50
A higher value means more relevant pages in the top 50.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis

The unit of analysis is an individual **webpage**.

Each row in the dataset represents one webpage and contains observable search, engagement, and content-related signals for that page.

For this lane, the relevant signals include impressions, average position, CTR, content age, days since last update, and word count.

The model will rank webpages based on these observable signals to help determine which pages deserve review first.

In [7]:
import pandas as pd

DATA_PATH = "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

lane_slice = df[
    [
        "content_id",
        "impressions_90d",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "word_count",
        "trend_direction"
    ]
].copy()

print("Rows:", len(lane_slice))
print("Columns:", len(lane_slice.columns))
print("One row = one webpage")

lane_slice.head()

Rows: 30000
Columns: 8
One row = one webpage


,content_id,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,trend_direction
0,content_304f48230142,3803,10.6,0.76,187,20,3221.0,down
1,content_a1fb4e703a9e,15320,20.3,0.05,445,25,2481.0,down
2,content_9aa793d4d895,12581,36.5,0.09,141,20,3515.0,down
3,content_331d6c4de07b,11751,6.2,0.49,463,22,NaN,stable
4,content_d99b7a2d90ca,19140,44.0,0.13,263,14,2803.0,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML Beats a Fixed Rule

A fixed rule can capture a simple pattern, such as prioritizing webpages that are both old and highly visible. However, the relationship between observable search and engagement signals is more complex than a single if-statement.

Different combinations of impressions, average position, CTR, content age, update recency, and other signals may be associated with different levels of declining trends.

Machine learning can learn patterns across multiple observable signals and combine them when ranking webpages. This can provide a more flexible prioritization than a single hand-written rule.

The goal is not to replace human judgment or prove that a page should be refreshed, but to provide a more evidence-based ranking for review.

In [8]:
features = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

print("Observable signals used:")
for feature in features:
    print("-", feature)

print("\nNumber of observable signals:", len(features))

Observable signals used:
- impressions_90d
- avg_position
- ctr
- content_age_days
- days_since_last_update
- word_count

Number of observable signals: 6


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.